# 🏥 Hospital Obstacle Avoidance - Cleaned Notebook

In [ ]:

# === IMPORTS ===
import os, re, glob, cv2, heapq, torch, numpy as np
from ultralytics import YOLO
from transformers import DPTForDepthEstimation, DPTImageProcessor

# === PATHS ===
BASE_CORRIDOR = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor"
VID_DIR       = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos"
FAIL_DIR      = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\failure_frames"
os.makedirs(VID_DIR, exist_ok=True)
os.makedirs(FAIL_DIR, exist_ok=True)

# === DEVICE & MODELS ===
device = "cuda" if torch.cuda.is_available() else "cpu"
yolo = YOLO("yolov8n.pt")

_depth_ready = False
def _ensure_depth_model():
    global _depth_ready, depth_model, depth_proc
    if _depth_ready: return
    depth_model = DPTForDepthEstimation.from_pretrained("Intel/dpt-hybrid-midas").to(device)
    depth_proc  = DPTImageProcessor.from_pretrained("Intel/dpt-hybrid-midas")
    _depth_ready = True

# === PARAMETERS ===
EXP_START, EXP_END = 400, 733
TEST_ONLY          = False

DET_CONF        = 0.32
MIN_AREA_FRAC   = 0.0015
MAX_AREA_FRAC   = 0.55
INFLATE_PIXELS  = 14
REPLAN_EVERY    = 4

USE_DEPTH_PROBE  = True
PROBE_EVERY      = 1
PROBE_WIDTH_FRAC = 0.40
PROBE_TARGET_W   = 448
ADAPT_PERCENTILE = 65
DILATE_ITER      = 3

AGG_FRAMES          = 60
AGG_DET_CONF        = 0.28
AGG_MIN_AREA_FRAC   = 0.0010
AGG_MAX_AREA_FRAC   = 0.60
AGG_INFLATE_PIXELS  = 12
AGG_PROBE_EVERY     = 1
AGG_PROBE_WIDTH     = 0.38
AGG_ADAPT_PERCENT   = 70

STOP_BLOCKED_THRESH = 0.14

# === HELPERS ===
def yolo_obstacles(frame_bgr, conf=DET_CONF):
    res = yolo.predict(source=frame_bgr, conf=conf, verbose=False,
                       device=0 if device=="cuda" else None)[0]
    H, W = frame_bgr.shape[:2]; A = H * W
    boxes = []
    if res.boxes is not None:
        xyxy = res.boxes.xyxy.cpu().numpy().astype(int)
        for (x1, y1, x2, y2) in xyxy:
            area = max(1, (x2-x1)*(y2-y1)) / A
            if area < MIN_AREA_FRAC or area > MAX_AREA_FRAC: continue
            boxes.append((x1, y1, x2, y2))
    return boxes

def boxes_to_mask(shape, boxes, inflate=INFLATE_PIXELS):
    H, W = shape[:2]; m = np.zeros((H, W), np.uint8)
    for (x1, y1, x2, y2) in boxes:
        x1i=max(0,x1-inflate); y1i=max(0,y1-inflate)
        x2i=min(W-1,x2+inflate); y2i=min(H-1,y2+inflate)
        m[y1i:y2i+1, x1i:x2i+1] = 255
    return m

def build_occupancy(mask, dilate_iter=2):
    grid = cv2.dilate(mask, np.ones((3,3), np.uint8), iterations=dilate_iter)
    return (grid>0).astype(np.uint8)

def astar(grid, start, goal):
    H,W = grid.shape
    def h(a,b): return abs(a[0]-b[0])+abs(a[1]-b[1])
    openq=[(h(start,goal),0,start,None)]; came,cost,seen={}, {start:0}, set()
    nbrs=[(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]
    while openq:
        _,g,cur,parent = heapq.heappop(openq)
        if cur in seen: continue
        seen.add(cur); came[cur]=parent
        if cur==goal: break
        for dx,dy in nbrs:
            nx,ny = cur[0]+dx, cur[1]+dy
            if not (0<=nx<H and 0<=ny<W): continue
            if grid[nx,ny]==1: continue
            ng = g + (1.4 if dx and dy else 1.0)
            if ng < cost.get((nx,ny), 1e9):
                cost[(nx,ny)] = ng
                heapq.heappush(openq, (ng + h((nx,ny),goal), ng, (nx,ny), cur))
    if goal not in came: return []
    path,node=[],goal
    while node: path.append(node); node=came[node]
    return path[::-1]

def depth_probe_mask(frame_bgr):
    _ensure_depth_model()
    H, W = frame_bgr.shape[:2]
    cx1 = int(W*(0.5 - PROBE_WIDTH_FRAC/2)); cx2 = int(W*(0.5 + PROBE_WIDTH_FRAC/2))
    roi = frame_bgr[:, cx1:cx2]
    h, w = roi.shape[:2]
    tw = PROBE_TARGET_W; th = int(h * (tw / max(1, w)))
    roi_small = cv2.resize(roi, (tw, th))
    rgb = cv2.cvtColor(roi_small, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = depth_proc(images=rgb, return_tensors="pt").to(device)
        d = depth_model(**inputs).predicted_depth.squeeze().detach().cpu().numpy()
    d = (d - d.min()) / (d.max() - d.min() + 1e-8)
    d8 = 255 - (d * 255).astype(np.uint8)
    d8 = cv2.resize(d8, (cx2-cx1, H), interpolation=cv2.INTER_LINEAR)
    thr = np.percentile(d8, ADAPT_PERCENTILE)
    mask = (d8 >= thr).astype(np.uint8) * 255
    full = np.zeros((H, W), np.uint8); full[:, cx1:cx2] = mask
    full = cv2.dilate(full, np.ones((3,3), np.uint8), iterations=DILATE_ITER)
    return full

def edge_plane_mask(frame_bgr,
                    width_frac=PROBE_WIDTH_FRAC,
                    edge_min_density=0.02,
                    bright_min=110,
                    dilate_iter=3):
    H, W = frame_bgr.shape[:2]
    cx1 = int(W * (0.5 - width_frac / 2.0))
    cx2 = int(W * (0.5 + width_frac / 2.0))
    roi = frame_bgr[:, cx1:cx2]
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 40, 120)
    edge_density = (edges > 0).mean()
    mean_bright = gray.mean()
    if edge_density < edge_min_density and mean_bright >= bright_min:
        m = np.zeros((H, W), np.uint8)
        m[:, cx1:cx2] = 255
        m = cv2.dilate(m, np.ones((3,3), np.uint8), iterations=dilate_iter)
        return m
    return np.zeros((H, W), np.uint8)

def path_hits_obstacle(grid, path, k=40):
    for r, c in path[:k]:
        if grid[r, c] == 1:
            return True
    return False

# === MAIN LOOP ===
exp_range = range(EXP_START, EXP_END) if not TEST_ONLY else range(EXP_START, min(EXP_START+6, EXP_END))

for exp in exp_range:
    exp_dir = rf"{BASE_CORRIDOR}\exp{exp}"
    out_path = rf"{VID_DIR}\exp{exp}_avoidance.mp4"

    if not os.path.isdir(exp_dir):
        print(f"skip (missing): {exp_dir}")
        continue

    frames = sorted(glob.glob(rf"{exp_dir}\exp{exp}_img*.jpg"), key=lambda x: int(re.search(r"_img(\d+)\.jpg$", x).group(1)))
    if not frames:
        print(f"no frames in {exp_dir}")
        continue

    first = cv2.imread(frames[0])
    H, W = first.shape[:2]
    vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 20.0, (W, H))
    print(f"exp{exp}: {len(frames)} frames")

    last_path = []
    for i, fp in enumerate(frames, 1):
        frame = cv2.imread(fp)
        if frame is None:
            continue

        boxes = yolo_obstacles(frame, conf=DET_CONF)
        mask = boxes_to_mask(frame.shape, boxes, inflate=INFLATE_PIXELS)

        if USE_DEPTH_PROBE and (i % PROBE_EVERY == 0):
            mask = cv2.bitwise_or(mask, depth_probe_mask(frame))

        e_mask = edge_plane_mask(frame)
        mask = cv2.bitwise_or(mask, e_mask)

        grid = build_occupancy(mask, dilate_iter=2)
        start = (int(H * 0.85), int(W * 0.5))
        goal = (int(H * 0.15), int(W * 0.5))
        if not last_path or (i % REPLAN_EVERY == 0):
            last_path = astar(grid, start, goal)

        bottom = grid[int(H*0.80):, int(W*0.40):int(W*0.60)]
        blocked = (bottom.mean() > STOP_BLOCKED_THRESH)

        vis = frame.copy()
        if blocked:
            cv2.putText(vis, "STOP: BLOCKED AHEAD", (10, H-20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            last_path = []

        if e_mask.any():
            red_overlay = np.zeros_like(vis)
            red_overlay[:, :, 2] = e_mask
            vis = cv2.addWeighted(vis, 1.0, red_overlay, 0.35, 0)

        for (x1, y1, x2, y2) in boxes:
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)
        for (r, c) in last_path:
            cv2.circle(vis, (c, r), 2, (0, 255, 0), -1)

        if last_path and path_hits_obstacle(grid, last_path):
            fname = os.path.join(FAIL_DIR, f"exp{exp}_frame{i:04d}_pathhit.png")
            overlay = cv2.applyColorMap(mask, cv2.COLORMAP_JET)
            debug = cv2.addWeighted(vis, 0.6, overlay, 0.4, 0)
            cv2.putText(debug, "PATH HIT OBSTACLE", (10, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            cv2.imwrite(fname, debug)

        cv2.putText(vis, f"exp{exp} frame {i}/{len(frames)} boxes:{len(boxes)} path:{len(last_path)}",
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        vw.write(vis)

    vw.release()
    print(f"✅ saved → {out_path}")


exp400: 229 frames
✅ saved → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp400_avoidance.mp4
exp401: 170 frames
✅ saved → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp401_avoidance.mp4
exp402: 210 frames
